In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
import mlflow
from mlflow import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

catalog = dbutils.widgets.get("catalog")
model_name = f"{catalog}.ml.fraud_model"
user = spark.sql("SELECT current_user()").first()[0]

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{user}/claims_fraud")
fe = FeatureEngineeringClient()
client = MlflowClient()

In [0]:
num = ["policy_age_days", "estimate_amount", "estimate_to_value", "estimate_vs_cause_avg",
       "workshop_claims_30d", "workshop_new_policy_share_30d", "is_tpa"]
cat = ["plan", "cause"]

labels = spark.table(f"{catalog}.bronze.investigations").select(
    "claim_id", (F.col("outcome") == "fraud").cast("int").alias("label")
)

training_set = fe.create_training_set(
    df=labels,
    feature_lookups=[FeatureLookup(
        table_name=f"{catalog}.ml.claim_features",
        lookup_key="claim_id",
        feature_names=num + cat,
    )],
    label="label",
)

df = training_set.load_df().toPandas().sort_values("claim_id").reset_index(drop=True)
X, y = df[num + cat], df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(len(X_train), len(X_test), y.sum(), y_test.sum())

In [0]:
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="NONE"),
                          OneHotEncoder(handle_unknown="ignore")), cat),
])

models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "gradient_boosting": HistGradientBoostingClassifier(random_state=42),
}

results = {}
for name, clf in models.items():
    with mlflow.start_run(run_name=name) as run:
        pipe = Pipeline([("pre", clone(pre)), ("clf", clf)]).fit(X_train, y_train)
        score = pipe.predict_proba(X_test)[:, 1]
        pred = (score >= 0.5).astype(int)
        metrics = {
            "pr_auc": average_precision_score(y_test, score),
            "roc_auc": roc_auc_score(y_test, score),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
        }
        mlflow.log_params({"model": name, "train_rows": len(X_train), "test_rows": len(X_test)})
        mlflow.log_metrics(metrics)
        fe.log_model(
            model=pipe,
            artifact_path="model",
            flavor=mlflow.sklearn,
            training_set=training_set,
            registered_model_name=model_name,
        )
        version = [v for v in client.search_model_versions(f"name='{model_name}'")
                   if v.run_id == run.info.run_id][0].version
        results[name] = (metrics, version)

In [0]:
ranked = sorted(results.items(), key=lambda r: r[1][0]["pr_auc"], reverse=True)
client.set_registered_model_alias(model_name, "champion", ranked[0][1][1])
client.set_registered_model_alias(model_name, "challenger", ranked[1][1][1])

for name, (m, v) in ranked:
    print(name, f"v{v}", {k: round(x, 3) for k, x in m.items()})